In [25]:
import random
random.seed(42)

import torch

import numpy as np
import pyomo.environ as pyo
import pandas as pd
import matplotlib.pyplot as plt

from pyomo.environ import *
from utils import generate_simple_synthetic_data, const_embed, load_case_study_data
from trainer import Trainer

np.random.seed(43)

LOAD_CASE_STUDY = False

In [26]:
if not LOAD_CASE_STUDY:
    X, y, true_mean, true_std = generate_simple_synthetic_data(n_samples=1000)

    print(X.head())
    print(y.head())
else:
    print("Loading case study data...")
    X, y, val_ind, test_ind = load_case_study_data()



             x1            x2          price
0  0.3745401188  0.1851329288  15.2341136747
1  0.9507143064  0.5419009474  14.9395759814
2  0.7319939418  0.8729458359  28.1250916104
3  0.5986584842  0.7322248864  14.9909239970
4  0.1560186404  0.8065611479  15.4389945226
0    4.1641150818
1    9.1615406641
2    0.4333924942
3    5.6618455395
4    5.2185912388
Name: y, dtype: float64


In [27]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

'''model_config = {
    'hidden_layers': 1,
    'hidden_size': 5,
    'drop': 0.2
    }

training_config = {
    'lr': 0.01,
    'wd': 0.0001,
    'batchsize': 64,
    'iters': 500,
    'patience': 20
    }'''

config = {
    # Model config
    'hidden_layers': 0,
    'hidden_size': 50,
    'drop': 0.05,

    # Training config
    'lr': 0.001,
    'wd': 0,
    'batchsize': 64,
    'iters': 5000,
    'patience': 500
}

trainer = Trainer(X, y, config, device=device)

if LOAD_CASE_STUDY:
    trainer.set_data_partitions(X_train=X[(~X.index.isin(val_ind)) & (~X.index.isin(test_ind))], y_train=y[(~y.index.isin(val_ind)) & (~y.index.isin(test_ind))], X_val=X[X.index.isin(val_ind)], y_val=y[y.index.isin(val_ind)], X_test=X[X.index.isin(test_ind)], y_test=y[y.index.isin(test_ind)])
else:
    trainer.create_data_partitions(val_size=0.2, test_size=0.2, random_state=42)

print("X_train shape: ", trainer.X_train.shape)
print("X_val shape: ", trainer.X_val.shape)
print("X_test shape: ", trainer.X_test.shape)

trainer.train()

trainer.test()

X_train shape:  (600, 3)
X_val shape:  (200, 3)
X_test shape:  (200, 3)
Validation loss improved from inf to 9821.1572 in epoch 1. Saving model...
Epoch 1/5000 - Train Loss: 61346.1048, Val Loss: 9821.1572
Validation loss improved from 9821.1572 to 8.4246 in epoch 2. Saving model...
Validation loss improved from 8.4246 to 4.4977 in epoch 3. Saving model...
Validation loss improved from 4.4977 to 3.7120 in epoch 4. Saving model...
Validation loss improved from 3.7120 to 3.4550 in epoch 5. Saving model...
Validation loss improved from 3.4550 to 3.3576 in epoch 6. Saving model...
Validation loss improved from 3.3576 to 3.3149 in epoch 7. Saving model...
Validation loss improved from 3.3149 to 3.2922 in epoch 8. Saving model...
Validation loss improved from 3.2922 to 3.2762 in epoch 9. Saving model...
Validation loss improved from 3.2762 to 3.2606 in epoch 10. Saving model...
Validation loss improved from 3.2606 to 3.2431 in epoch 11. Saving model...
Epoch 11/5000 - Train Loss: 4.0048, Val

In [28]:
trainer.model

distModel(
  (input_layer): Linear(in_features=3, out_features=50, bias=True)
  (hidden_layers): ModuleList()
  (output_mean_layer): Linear(in_features=50, out_features=1, bias=True)
  (output_sd_layer): Linear(in_features=50, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.05, inplace=False)
)

In [29]:
#trainer.predict(X=trainer.X_test)

In [30]:
trainer.evaluate_gaussian_nll()
trainer.evaluate_standardized_residuals()

Baseline Gaussian NLL Loss: 0.4348
Model Gaussian NLL Loss: -0.9925
KS test against N(0,1): statistic = 0.0896, p-value = 0.0758
✅ The standardized residuals do not significantly differ from a standard normal distribution — model variance estimates appear well-calibrated.
Saved: residual_diagnostics\z_histogram.png
Saved: residual_diagnostics\z_qqplot.png


(np.float64(0.08959054354084939), np.float64(0.07581196982910754))

In [31]:
'''# Inputs
x = np.array([[-1.266313, -0.405118, 0.612491]])

# Extract weights from model
state_dict = trainer.model.state_dict()
w0 = state_dict['input_layer.weight'].cpu().numpy()
b0 = state_dict['input_layer.bias'].cpu().numpy()

w1 = state_dict['hidden_layers.0.weight'].cpu().numpy()
b1 = state_dict['hidden_layers.0.bias'].cpu().numpy()

w_mean = state_dict['output_mean_layer.weight'].cpu().numpy()
b_mean = state_dict['output_mean_layer.bias'].cpu().numpy()

w_sd = state_dict['output_sd_layer.weight'].cpu().numpy()
b_sd = state_dict['output_sd_layer.bias'].cpu().numpy()

# Manual forward pass
def relu(x): return np.maximum(0, x)

h0 = relu(x.dot(w0.T) + b0)
h1 = relu(h0.dot(w1.T) + b1)
mean = h1.dot(w_mean.T) + b_mean
sd = relu(h1.dot(w_sd.T) + b_sd)

print("Manual mean:", mean)
print("Manual std:", sd)'''

'# Inputs\nx = np.array([[-1.266313, -0.405118, 0.612491]])\n\n# Extract weights from model\nstate_dict = trainer.model.state_dict()\nw0 = state_dict[\'input_layer.weight\'].cpu().numpy()\nb0 = state_dict[\'input_layer.bias\'].cpu().numpy()\n\nw1 = state_dict[\'hidden_layers.0.weight\'].cpu().numpy()\nb1 = state_dict[\'hidden_layers.0.bias\'].cpu().numpy()\n\nw_mean = state_dict[\'output_mean_layer.weight\'].cpu().numpy()\nb_mean = state_dict[\'output_mean_layer.bias\'].cpu().numpy()\n\nw_sd = state_dict[\'output_sd_layer.weight\'].cpu().numpy()\nb_sd = state_dict[\'output_sd_layer.bias\'].cpu().numpy()\n\n# Manual forward pass\ndef relu(x): return np.maximum(0, x)\n\nh0 = relu(x.dot(w0.T) + b0)\nh1 = relu(h0.dot(w1.T) + b1)\nmean = h1.dot(w_mean.T) + b_mean\nsd = relu(h1.dot(w_sd.T) + b_sd)\n\nprint("Manual mean:", mean)\nprint("Manual std:", sd)'

In [32]:
cons = trainer.constraint_build()
cons.to_csv('constraints.csv')

In [33]:
cons = pd.read_csv('constraints.csv', index_col=0)
pd.set_option('display.precision', 10)
cons

,intercept,layer,node,node_0,node_1,node_2,node_3,node_4,node_5,node_6,...,node_40,node_41,node_42,node_43,node_44,node_45,node_46,node_47,node_48,node_49
0,0.6779146000,0,0,-0.188254100,-0.638588600,-0.172332930,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.2116805300,0,1,0.386523070,-0.110227740,-0.287262950,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.4024943400,0,2,0.058709030,0.038973458,-0.097894270,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.4906951800,0,3,0.147300470,-0.266274630,-0.254659400,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-0.2432052800,0,4,-0.166126530,0.225335700,-0.039615393,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0.2330293400,0,5,-0.530603230,-0.115756400,0.009267576,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.3960148000,0,6,-0.317019370,-0.130347010,0.129868790,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.5506942000,0,7,0.435406660,0.332051600,-0.261411250,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.3613306000,0,8,-0.422160450,0.253077660,0.064485540,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0.3473700300,0,9,-0.286944750,0.267391400,-0.500320200,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:

'''#contextual_var = trainer.X_test
contextual_var = trainer.X_test.iloc[[0]]
contextual_var.reset_index(drop=True, inplace=True)
contextual_var'''

'#contextual_var = trainer.X_test\ncontextual_var = trainer.X_test.iloc[[0]]\ncontextual_var.reset_index(drop=True, inplace=True)\ncontextual_var'

In [35]:
'''row = pd.DataFrame([{
    'x1': -1.266313,
    'x2': -0.405118,
    'price': 30.0
}])

def preict_one_row(row):
    # Predict
    mean_pred, sd_pred = trainer.predict(row, unscale=True)
    print("Mean prediction:", mean_pred)
    print("Standard deviation prediction:", sd_pred)

preict_one_row(row)'''

'row = pd.DataFrame([{\n    \'x1\': -1.266313,\n    \'x2\': -0.405118,\n    \'price\': 30.0\n}])\n\ndef preict_one_row(row):\n    # Predict\n    mean_pred, sd_pred = trainer.predict(row, unscale=True)\n    print("Mean prediction:", mean_pred)\n    print("Standard deviation prediction:", sd_pred)\n\npreict_one_row(row)'

In [36]:
def model_definition(contextual_var, n_sc, n_preds):
    model = pyo.ConcreteModel()
    
    model.N = pyo.RangeSet(0,n_preds-1)
    model.W = pyo.RangeSet(1,n_sc) # scenarios

    pi = [1/n_sc for i in range(n_sc)]

    model.var_ind = pyo.Set(initialize=contextual_var.columns.sort_values())

    model.x = pyo.Var(model.var_ind, model.N) # price

    model.y = pyo.Var(pyo.Any, dense=False, domain=pyo.Reals)

    model.rent = pyo.Var(model.W, within=pyo.Reals) # rentabilidad per scenario

    #definition of the objective function
    def obj_expression(model):
        return sum(pi[w-1] * model.rent[w] for w in model.W)
    model.OBJ = pyo.Objective(rule=obj_expression, sense=pyo.maximize)

    #definition of the constraints
    def rent_expression(model, w):
        return model.rent[w] == sum(model.y['demand', n, w]*model.x['price',n] for n in model.N)
    model.rent_expression = pyo.Constraint(model.W, rule=rent_expression)

    #constraint for fixing contextual information
    def const9_rule(model,x_ind,n):
        if x_ind=='price':
            return pyo.Constraint.Skip
        else:
            return model.x[x_ind,n] == contextual_var.loc[n,x_ind]
    model.const9 = pyo.Constraint(model.var_ind, model.N, rule=const9_rule)

    P_max = 30
    # max price constraint for each price
    def max_price_expression(model, n):
        return model.x['price', n] <= P_max

    model.max_price_expression = pyo.Constraint(model.N, rule=max_price_expression)

    def min_price_expression(model, n):
        return model.x['price', n] >= 0
    model.min_price_expression = pyo.Constraint(model.N, rule=min_price_expression)

    return model


In [37]:
n_preds = 2
n_sc =  30

contextual_var = trainer.X[trainer.X.index.isin(trainer.test_indices)]
contextual_var = contextual_var.iloc[:n_preds]
contextual_var.reset_index(drop=True, inplace=True)
contextual_var

,price,x1,x2
0,15.2341136747,0.3745401188,0.1851329288
1,28.1250916104,0.7319939418,0.8729458359


In [38]:
'''n_preds = 3
n_sc =  30

contextual_var = trainer.X_test.iloc[:n_preds]
contextual_var.reset_index(drop=True, inplace=True)
contextual_var'''

'n_preds = 3\nn_sc =  30\n\ncontextual_var = trainer.X_test.iloc[:n_preds]\ncontextual_var.reset_index(drop=True, inplace=True)\ncontextual_var'

In [39]:
model = model_definition(contextual_var, n_sc, n_preds)
const_embed(model=trainer, opt_model=model, constaints=cons, outcome='demand', n_scenarios=n_sc, n_preds=n_preds)

In [40]:
opt = pyo.SolverFactory('gurobi_direct')
opt.options['threads'] = 0
opt.options['NonConvex'] = 2
results = opt.solve(model, tee=True)

Set parameter OutputFlag to value 1
Set parameter Threads to value 0
Set parameter NonConvex to value 2
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-13500, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
NonConvex  2

Optimize a model with 372 rows, 300 columns and 1492 nonzeros
Model fingerprint: 0x1a3f5298
Model has 30 quadratic constraints
Variable types: 200 continuous, 100 integer (100 binary)
Coefficient statistics:
  Matrix range     [7e-04, 1e+03]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [3e-02, 3e-02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-02, 1e+03]
Presolve removed 95 rows and 92 columns
Presolve time: 0.12s
Presolved: 398 rows, 270 columns, 1188 nonzeros
Presolved model has 60 bilinear constraint(s)

Solving non-convex MIQCP

Variable types: 199 c

In [41]:
print(value(model.x['price', 0]))
print((value(model.x['price', 0]) * trainer.X_std[0]) + trainer.X_mean[0])
print((value(model.x['price', 0] - trainer.X_mean[0])) / trainer.X_std[0])

10.283496730540044
79.51160113794145
-1.7020796172793466


In [42]:
for c in model.component_objects(Constraint, active=True):
    print(f"Constraint component: {c.name}")
    for index in c:
        print(f"  {index}: {c[index].expr}")

Constraint component: rent_expression
  1: rent[1]  ==  y[demand,0,1]*x[price,0] + y[demand,1,1]*x[price,1]
  2: rent[2]  ==  y[demand,0,2]*x[price,0] + y[demand,1,2]*x[price,1]
  3: rent[3]  ==  y[demand,0,3]*x[price,0] + y[demand,1,3]*x[price,1]
  4: rent[4]  ==  y[demand,0,4]*x[price,0] + y[demand,1,4]*x[price,1]
  5: rent[5]  ==  y[demand,0,5]*x[price,0] + y[demand,1,5]*x[price,1]
  6: rent[6]  ==  y[demand,0,6]*x[price,0] + y[demand,1,6]*x[price,1]
  7: rent[7]  ==  y[demand,0,7]*x[price,0] + y[demand,1,7]*x[price,1]
  8: rent[8]  ==  y[demand,0,8]*x[price,0] + y[demand,1,8]*x[price,1]
  9: rent[9]  ==  y[demand,0,9]*x[price,0] + y[demand,1,9]*x[price,1]
  10: rent[10]  ==  y[demand,0,10]*x[price,0] + y[demand,1,10]*x[price,1]
  11: rent[11]  ==  y[demand,0,11]*x[price,0] + y[demand,1,11]*x[price,1]
  12: rent[12]  ==  y[demand,0,12]*x[price,0] + y[demand,1,12]*x[price,1]
  13: rent[13]  ==  y[demand,0,13]*x[price,0] + y[demand,1,13]*x[price,1]
  14: rent[14]  ==  y[demand,0,14]*x

In [43]:
# Inputs
x = np.array([[-1.266313, -0.405118, 0.612491]])

# Extract weights from model
state_dict = trainer.model.state_dict()
w0 = state_dict['input_layer.weight'].cpu().numpy()
b0 = state_dict['input_layer.bias'].cpu().numpy()

#w1 = state_dict['hidden_layers.0.weight'].cpu().numpy()
#b1 = state_dict['hidden_layers.0.bias'].cpu().numpy()

w_mean = state_dict['output_mean_layer.weight'].cpu().numpy()
b_mean = state_dict['output_mean_layer.bias'].cpu().numpy()

w_sd = state_dict['output_sd_layer.weight'].cpu().numpy()
b_sd = state_dict['output_sd_layer.bias'].cpu().numpy()

# Manual forward pass
def relu(x): return np.maximum(0, x)

h0 = relu(x.dot(w0.T) + b0)
#h1 = relu(h0.dot(w1.T) + b1)
mean = h0.dot(w_mean.T) + b_mean
sd = relu(h0.dot(w_sd.T) + b_sd)

print("Manual mean:", mean)
print("Manual std:", sd)

Manual mean: [[1.16346762]]
Manual std: [[0.32203757]]


In [44]:
contextual_var

,price,x1,x2
0,15.2341136747,0.3745401188,0.1851329288
1,28.1250916104,0.7319939418,0.8729458359


In [45]:
print((15.2341136747 - trainer.X_mean[0]) / trainer.X_std[0])
print((0.3745401188 - trainer.X_mean[1]) / trainer.X_std[1])
print((0.1851329288 - trainer.X_mean[2]) / trainer.X_std[2])

-0.8449711082285967
-0.425561705775379
-1.1629965028645957


In [46]:
row = pd.DataFrame([{
    'price': 14.838617787184095,
    'x1': 0.7319939418,
    'x2': 0.8729458359,
}])

row = row.reindex(sorted(row.columns), axis=1)

def preict_one_row(row):
    # Predict
    mean_pred, sd_pred = trainer.predict(row, unscale=False, scale_data=True)
    print("Mean prediction:", mean_pred)
    print("Standard deviation prediction:", sd_pred)

preict_one_row(row)

Scaling data...
Mean prediction: [[1.5440329]]
Standard deviation prediction: [[0.34075832]]


In [47]:
for n in model.N:
    print("&"*50)
    print("Establecimiento ", n)
    opt_price = value(model.x['price',n])
    total_demand = 0
    for x in range(n_sc):
        total_demand += value(model.y['demand', n, x+1])
        #print("demand: ", value(model.y['demand', 0, x+1]))
    print("Average demand: ", total_demand/n_sc)
    opt_mean = value(model.y['demand', n,'mean'])
    opt_sd = value(model.y['demand', n,'sd'])
    opt_profit = value(model.OBJ)

    print(f"Optimal profit: {opt_profit}")
    print(f"Optimal price: {opt_price}")
    print(f"Optimal mean: {opt_mean}")
    print(f"Optimal sd: {opt_sd}")
    print("&"*50)

&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
Establecimiento  0
Average demand:  6.198601406377272
Optimal profit: 185.46419920531778
Optimal price: 10.283496730540044
Optimal mean: 0.9293199046950089
Optimal sd: 0.2993529258543465
&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
Establecimiento  1
Average demand:  9.262561085583393
Optimal profit: 185.46419920531778
Optimal price: 13.141171300707246
Optimal mean: 1.919147313813751
Optimal sd: 0.3085844830622695
&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&&
